<a href="https://colab.research.google.com/github/Lil0nly2/DataScience/blob/main/1_data_warehouse_dimensional_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1: Data Warehousing, Dimensional Modeling & Star Schemas

Welcome to  **Data Warehousing & Dimensional Modeling** notebook.

Modern data analytics relies on structured data models designed specifically for high-speed reporting, business intelligence (BI), and aggregated queries. In this module, we bridge the gap between operational databases and analytical data warehouses.

---

### 🗺️ Learning Roadmap
1. **Step 0: OLTP vs. OLAP**: Why operational databases fail at analytics and why we build Data Warehouses
2. **Step 1: Primary Keys (PK) & Foreign Keys (FK)**: Enforcing referential integrity and identity

3. **Step 3: Dimension Tables (`dim_...`)**: The "Context" — Who, What, Where, When, and Why
4. **Step 4: Fact Tables (`fact_...`)**: The "Heartbeat" — Business events, numeric measures, and defining the **Grain**
5. **Step 5: Architectural Patterns**: **Star Schema** vs. **Snowflake Schema**
---

## ⚙️ Step 0: The Core Paradigm — OLTP vs. OLAP

### Mental Model: Running the Business vs. Understanding the Business

| Feature | **OLTP** (Online Transaction Processing) | **OLAP** (Online Analytical Processing) |
| :--- | :--- | :--- |
| **Primary Goal** | Fast, reliable transaction execution (write-heavy) | High-speed analytical queries & reporting (read-heavy) |
| **Typical Operations**| `INSERT`, `UPDATE`, `DELETE` of single records | Heavy `SUM()`, `AVG()`, `COUNT()` with multiple joins |
| **Query Scope** | Fetching single customer order or updating status | Aggregating 10 million transactions across 3 years |
| **Examples** | PostgreSQL, MySQL, SQLite, Oracle (App backend) | Snowflake, Google BigQuery, Amazon Redshift, DuckDB |


## Key Reasons to Use Normalization

1. ### Eliminating Data Redundancy
Without normalization, the same piece of information (like a customer's physical address) might be repeated across dozens of rows for every order they place. Normalization ensures each fact is stored in only one place, saving storage space and keeping the database clean.
### Unnormalized Database

| OrderID | CustomerName | CustomerAddress | OrderDate | TotalAmount |
| :--- | :--- | :--- | :--- | :--- |
| 1001 | Alice Smith | 123 Maple St, New York | 2026-10-01 | \$150.00 |
| 1002 | Bob Jones | 456 Oak Ave, Los Angeles | 2026-10-02 | \$45.00 |
| 1003 | Alice Smith | 123 Maple St, New York | 2026-10-10 | \$89.00 |
| 1004 | Alice Smith | 123 Maple St, New York | 2026-10-12 | \$210.00 |

---

### Normalized Database

#### 1. Customers Table

| CustomerID (PK) | CustomerName | CustomerAddress |
| :--- | :--- | :--- |
| 1 | Alice Smith | 123 Maple St, New York |
| 2 | Bob Jones | 456 Oak Ave, Los Angeles |

#### 2. Orders Table

| OrderID | CustomerID (FK) | OrderDate | TotalAmount |
| :--- | :--- | :--- | :--- |
| 1001 | 1 | 2026-10-01 | \$150.00 |
| 1002 | 2 | 2026-10-02 | \$45.00 |
| 1003 | 1 | 2026-10-10 | \$89.00 |
| 1004 | 1 | 2026-10-12 | \$210.00 |

```text
+---------------------------------------+        ETL Pipeline        +-----------------------------------------+
|      OLTP Database (Normalized)       |   ===========================>   |       OLAP Data Warehouse (Star)        |
|  - 30+ normalized tables              |   (Extract, Transform, Load)     |  - Central Fact Table (Measures)        |
|  - Optimized for zero redundancy      |                                  |  - Surrounding Dimension Tables (Context)|
+---------------------------------------+                                  +-----------------------------------------+
```

In [1]:
import sqlite3
import pandas as pd

# 1. Initialize an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# 2. CRITICAL in SQLite: Enable Foreign Key constraint enforcement
# (By default in SQLite, foreign keys are disabled for backwards compatibility)
conn.execute("PRAGMA foreign_keys = ON;")

# 3. Helper function to execute SQL queries and return clean DataFrames
def run_query(sql: str) -> pd.DataFrame:
    """Execute SQL and render as a Pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

print("Environment initialized! Foreign key enforcement: ACTIVE")

Environment initialized! Foreign key enforcement: ACTIVE


---

## 🔑 Step 1: Primary Keys (PK) & Foreign Keys (FK)

### 1. Primary Key (`PRIMARY KEY`)
A Primary Key is a column (or group of columns) that **uniquely identifies every single row** in a table.
- **Uniqueness**: No two rows can have the same PK value.
- **Non-Nullability**: A PK can never be `NULL`.
- **Single vs. Composite**: A single PK uses one column (`customer_id`); a composite PK combines multiple columns (`order_id, line_item_id`).

### 2. Foreign Key (`FOREIGN KEY`)
A Foreign Key is a column in a child table that **points directly to the Primary Key of a parent table**.
- **Referential Integrity**: The database guarantees that you cannot insert a child row with an invalid foreign key that does not exist in the parent table.

```text
Parent Table: categories                     Child Table: products
+-----------------------------+              +------------------------------------------------+
| category_id (PK) | cat_name |              | product_id (PK) | product_name | category_id(FK) |
+------------------+----------+              +-----------------+--------------+-----------------+
|        1         | Electronics <---------------\             | Laptop       |        1        |
|        2         | Apparel  |<------------------+------------| Headphones   |        1        |
+-----------------------------+                   \------------| T-Shirt      |        2        |
                                                               +--------------------------------+
```

---

## 🏛️ Step 2: Dimension Tables (`dim_...`) — The Context

### Mental Model: The "Who, What, Where, When, and Why"
Dimension tables contain descriptive, categorical context that business analysts use to **filter (`WHERE`)**, **group (`GROUP BY`)**, and **slice** their reports.

### Characteristics of Dimension Tables:
1. **Wide & Shallow**: Many descriptive textual columns (20 to 50+ attributes), but relatively few rows (thousands to hundreds of thousands).
2. **Denormalized**: We deliberately avoid breaking cities, states, and countries into separate lookup tables. We keep them together in one flat `dim_customer` table to eliminate unnecessary join hops!
3. **Contains Hierarchies**: e.g., `Day -> Month -> Quarter -> Year` or `Product -> Subcategory -> Category`.

### Common Dimension Tables in Any Business:
- `dim_date`: Date hierarchy (`date_key`, `calendar_date`, `month_name`, `quarter`, `year`, `is_weekend`, `fiscal_period`).
- `dim_product`: Product hierarchy (`product_key`, `sku`, `product_name`, `brand`, `category`, `unit_cost`).
- `dim_customer`: Customer profile (`customer_key`, `customer_id`, `name`, `segment`, `city`, `state`).
- `dim_store`: Physical/online retail location (`store_key`, `store_name`, `store_type`, `region`, `city`).

---

## 📈 Step 3: Fact Tables (`fact_...`) — Events & Measurements

### Mental Model: The Business Heartbeat
Fact tables record the numeric measurements and metrics generated by a business process (sales, clicks, payments, shipments).

### 1. Defining the **Grain** (Most Critical Concept!)
The **Grain** is the fundamental atomic level of detail represented by a single row in the fact table.
> *"What does one row in this table represent?"*
- *Example Grain*: One row represents **one individual line-item on a sales receipt**.
- Never mix grains in a single fact table (e.g., mixing daily store summaries with individual transactions).

### 2. Anatomy of a Fact Table:
- **Foreign Keys (Dimension Keys)**: Columns pointing to surrounding dimension tables (`date_key`, `product_key`, `customer_key`, `store_key`).
- **Measures (Numeric Facts)**:
  - **Fully Additive**: Can be summed across all dimensions (e.g., `quantity_sold`, `sales_amount`, `cost`).
  - **Semi-Additive**: Can be summed across some dimensions, but not time (e.g., `account_balance` — you can't sum your bank balance across 30 days of the month).
  - **Non-Additive**: Ratios and percentages (e.g., `profit_margin_pct`, `unit_discount_rate`).

---

## 🌟 Step 4: Star Schema vs. Snowflake Schema

```text
                      STAR SCHEMA                                              SNOWFLAKE SCHEMA
             
             
                 +--------------+
                 |   dim_date   |
                 +-------+------+
                         |
                         v
  +--------------+  +----+----+  +--------------+              +--------------+  +----+----+  +--------------+
  | dim_customer |->|  FACT   |<-| dim_product  |              | dim_customer |->|  FACT   |<-| dim_product  |
  +--------------+  |  SALES  |  +--------------+              +--------------+  +----+----+  +-------+------+
                    +----+----+                                                                       |
                         ^                                                                            v
                         |                                                                   +--------+--------+
                 +-------+------+                                                            | dim_subcategory |
                 |  dim_store   |                                                            +--------+--------+
                 +--------------+                                                                     |
                                                                                                      v
                                                                                             +--------+--------+
                                                                                             |  dim_category   |
                                                                                             +-----------------+
```

| Criteria | **Star Schema (Industry Standard)** | **Snowflake Schema** |
| :--- | :--- | :--- |
| **Structure** | Fact in the center, 1-hop flat dimensions | Dimensions are broken down into sub-dimensions |
| **Join Complexity** | **Low** (Fact table joins directly to any dimension)| **High** (Requires multi-table join paths) |
| **Query Performance** | **Fastest** (fewer joins, optimal for columnar stores) | Slower (extra joins tax the query engine) |
| **BI Usability** | Very easy for analysts and tools (Power BI, Tableau)| Complex; confusing relationships for end users |